# **Durability emulator — dataset generation**

This notebook **only** runs the emulator and saves the resulting lambda datasets — no PCE is fitted
here. [`02_train_pce.ipynb`](02_train_pce.ipynb) only reads what this notebook writes.

The simulator is the closed-form carbonation model of Possan et al. (2016),
`carbonation_depth_possan_by_type` — no trained ML model is loaded.

**Artefacts written per time step**,
`<n_latent_samples>_<kind>_<split>_<t>_install_<year>_cement_<type>_exposure_<exposure>.pkl`:

| kind | split | content |
|---|---|---|
| `dataset_full` | train / val | one row per latent replica: inputs, effective values, carbonation depth, $g$, lambdas, processing time |
| `dataset_unique` | train / val | one row per design point: inputs, the four lambdas, processing time |

Plus one aggregate `<n_latent_samples>_emulator_timing_durability.pkl`, read back by
[`02_train_pce.ipynb`](02_train_pce.ipynb) to compute the emulator/surrogate speed-up.

CO₂ uses the published CMIP6/SSP table. Set `installation_year` and `co2_scenario` below.
The 100-year horizon must end no later than 2100. Regenerate datasets and retrain after changing the scenario; old polynomial results are not compatible.


## **1. Libraries**

In [1]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Uniform, JointIndependent

e:\github\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Random variables and fixed parameters**

Design variables (compressive strength, relative humidity, cover) plus everything the emulator
needs that isn't a design variable.

In [2]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

cement_type          = 3
installation_year    = 1980
co2_scenario         = "SSP2-4.5"  # SSP1-2.6, SSP2-4.5, or SSP5-8.5
exposure_conditions  = 2
ad                   = 0.0      # Pozzolanic material in the concrete, relative to cement mass (%) — Possan et al. (2016) model input.
                                 # cement_type=3 is CP II F (limestone filler, not pozzolanic), and there is no record of a
                                 # supplementary pozzolanic addition to this 1980 mix — ad=10 was inherited from matching the
                                 # old ML surrogate's training assumption, not from anything specific to this structure. ad
                                 # raises predicted carbonation depth by ~12% at any age (it depletes the alkaline reserve
                                 # faster), so ad=0 is the less conservative (slower-carbonation) of the two.
n_samples            = 2000      # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 500       # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
latent_cov           = 0.15    # CoV of the RH/fck/cover latent multipliers. 2% is low for real execution
                                 # variability (15-30%+ is routinely reported for cover/strength); fck and cover
                                 # are drawn Lognormal (can't go negative), RH is drawn Normal truncated to [0, 100]

## **3. Design samples**

In [3]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover: {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 2000
   Number of latent samples per design sample: 2500
   Total simulations per time step: 6250000

Sample statistics:
   fck:   20.0 - 50.0 MPa (mean: 34.8 MPa)
   RH:    20.1 - 80.0% (mean: 49.7%)
   cover: 15.0 - 60.0 mm (mean: 37.3 mm)


## **4. Time grid**

In [4]:
times = np.linspace(0, 100, 5, endpoint=True)  # Time points for carbonation depth prediction
times

array([  0.,  25.,  50.,  75., 100.])

## **5. Generate the dataset at each time step**

The same `x_val` (drawn once, in section 3) is reused across every time step — so the same
validation design points can be tracked over time, instead of being redrawn independently at each
`t`. `generate_dataset_at_time_durability` does the rest: latent sampling, carbonation depth
prediction, GLD fit, and saving `dataset_full`/`dataset_unique` for both splits.

In [5]:
print("="*60)
print("GENERATING THE DURABILITY DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_durability(
                                                    x_train=x_pce_rvs,
                                                    x_val=x_val,
                                                    time_step=t,
                                                    cement_type=cement_type,
                                                    installation_year=installation_year,
                                                    co2_scenario=co2_scenario,
                                                    exposure_conditions=exposure_conditions,
                                                    ad=ad,
                                                    n_latent_samples=n_latent_samples,
                                                    latent_cov=latent_cov,
                                                    output_dir='.',
                                                )
    generation_results.append(result)

GENERATING THE DURABILITY DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------
  train: 2000 design points, 275.89 s total
  val: 500 design points, 64.64 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 25.0 years
----------------------------------------
  train: 2000 design points, 262.32 s total
  val: 500 design points, 62.59 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 50.0 years
----------------------------------------
  train: 2000 design points, 249.75 s total
  val: 500 design points, 63.72 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 75.0 years
----------------------------------------
  train: 2000 design points, 274.38 s total
  val: 500 design points, 63.65 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 100.0 years
-------------------------------

## 6. Timing summary

Cost of building the dataset, per time step.

In [6]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

save_pkl(emulator_timing, f'{n_latent_samples}_emulator_timing_durability.pkl')

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - emulator g-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - emulator g-value dataset generation time: {val_total:.1f} s")
print(f"Total emulator g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - emulator g-value dataset generation time: 1307.8 s
Val split   - emulator g-value dataset generation time: 312.2 s
Total emulator g-value dataset generation time (train + val): 1620.0 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.0,2000,275.891445,137.945723,500,64.639119
1,25.0,2000,262.321094,131.160547,500,62.585168
2,50.0,2000,249.748146,124.874073,500,63.718679
3,75.0,2000,274.382576,137.191288,500,63.647124
4,100.0,2000,245.501047,122.750523,500,57.596140


## 7. Unique dataset statistics

Concatenate the `dataset_unique` (train + val, all time steps) frames already held in
`generation_results` and describe the columns, including the fitted lambdas.

In [7]:
dataset_unique_all = pd.concat(
    [
        result[f'df_unique_{split}'].assign(split=split)
        for result in generation_results
        for split in ('train', 'val')
    ],
    ignore_index=True,
)

print(f"Combined unique dataset: {len(dataset_unique_all)} rows "
      f"({len(generation_results)} time steps x train/val splits)")
dataset_unique_all.describe()

Combined unique dataset: 12500 rows (5 time steps x train/val splits)


,fck,rh,cov,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
count,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000
mean,34.868321,49.321558,37.402677,25.520339,0.244810,0.201702,0.136161,0.129602
std,8.672569,17.688171,12.996353,15.617798,0.098167,0.578907,0.590496,0.014884
min,20.031273,20.055871,15.018754,-32.657397,0.003487,-0.075082,-0.044414,0.085140
25%,27.137807,33.689550,26.109432,14.542114,0.176234,0.095787,0.058391,0.120867
50%,34.485558,48.980943,37.263493,25.451294,0.218111,0.158824,0.078391,0.128843
75%,42.389313,64.704392,48.745320,37.364516,0.289914,0.217455,0.106880,0.137609
max,49.979483,79.978561,59.967014,66.725132,0.671395,8.996268,8.782267,0.473826
